# Chunking strategy

Articles are split by HTML structure, not by length. There is no size limit and no overlap.

1. **Parse** (`DataProcessor.parse_html`): keep the text of `h1`–`h6`, `p`, `li`, `tr`, `blockquote`, and `pre` as ordered `(tag, text)` blocks. Skip `script` and `style`.
2. **Chunk** (`DataProcessor.chunk_blocks`): one chunk per block, except:
   - A heading is merged into the body block after it as `"Heading: body"`. A heading with no body after it becomes its own chunk.
   - A table row (`tr`) with no heading merged in gets its section prefixed, `"Section: row"`. Cells are joined with `": "`.
   - `chunk_id` is `{slug}_{index}`. `index` counts headings too, so IDs are unique but not contiguous.
3. **Save** (`DataProcessor.chunk_all_articles`): writes `plant_care_chunks` when `save_table=True`.

**Known structure loss**
- **Tables:** each `tr` is one chunk (153 tags, 153 chunks). Column headers are not tied to their cells.
- **Lists:** each `li` is its own chunk (347 tags, 347 chunks). The list grouping is lost.
- **Links:** `href` is dropped; the link text is kept.
- **Heading levels:** `section` keeps only the nearest heading, not its parent.
- **Untagged text:** 16 `text` chunks. Their source tag is not identified.

In [ ]:
import re
from collections import Counter

from loguru import logger
from plant_care.config import get_env, load_config
from plant_care.data_processor import DataProcessor
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()

# Load config
env = get_env()
cfg = load_config("../project_config.yml", env)

processor = DataProcessor(spark, cfg)

In [ ]:
# Chunk all parsed articles without saving, so the strategy can be tested first
chunks_df = processor.chunk_all_articles(save_table=True)
logger.info(f"Chunks: {chunks_df.count()}")

for row in chunks_df.limit(12).collect():
    print(f"[{row['tag']}] (section: {row['section']}) {row['text'][:150]}")

### Examples of HTML Elements and Chunks

In [ ]:
# Verify the tags that occur in the articles to understand structural loss
tags = ["ul", "ol", "li", "table", "tr", "td", "th", "a"]
rows = spark.table(processor.plant_table).filter("volume_path IS NOT NULL").select("volume_path").collect()

tag_total = Counter()
tag_articles = Counter()
for row in rows:
    with open(row["volume_path"], encoding="utf-8") as f:
        html = f.read()
    for t in tags:
        n = len(re.findall(rf"<{t}[\s>/]", html))
        tag_total[t] += n
        if n:
            tag_articles[t] += 1

print(f"Articles read: {len(rows)}")
for t in tags:
    print(f"<{t}>: {tag_total[t]} occurrences in {tag_articles[t]} articles")

chunks_df.groupBy("tag").count().orderBy("tag").show()

In [ ]:
sizes = chunks_df.select("chunk_id", "slug", "tag", F.length("text").alias("chars"))
sizes.select(F.min("chars").alias("min_chars"), F.max("chars").alias("max_chars"), F.avg("chars").alias("avg_chars")).show()

print("Smallest chunks:")
sizes.orderBy("chars").limit(5).show(truncate=False)
print("Largest chunks:")
sizes.orderBy(F.desc("chars")).limit(5).show(truncate=False)

In [ ]:
chunk = chunks_df.filter(F.col("chunk_id") == "bucephalandra_27").collect()[0]
volume_path = spark.table(processor.plant_table).filter(F.col("slug") == chunk["slug"]).select("volume_path").collect()[0]["volume_path"]
with open(volume_path, encoding="utf-8") as f:
    html = f.read()


def cell_texts(row):
    cells = re.findall(r"<t[dh]\b[^>]*>(.*?)</t[dh]>", row, flags=re.S)
    return [" ".join(re.sub(r"<[^>]+>", "", cell).split()) for cell in cells]


raw_rows = re.findall(r"<tr\b.*?</tr>", html, flags=re.S)
match = next(r for r in raw_rows if chunk["text"].endswith(": ".join(cell_texts(r))))
print("RAW HTML:", " ".join(match.split()))
print(f"CHUNK {chunk['chunk_id']} [{chunk['tag']}] section={chunk['section']!r}: {chunk['text']}")

In [ ]:
articles = spark.table(processor.plant_table).filter("volume_path IS NOT NULL").select("slug", "volume_path").collect()


def first_article_matching(pattern):
    for row in articles:
        with open(row["volume_path"], encoding="utf-8") as f:
            html = f.read()
        m = re.search(pattern, html, flags=re.S)
        if m:
            return row["slug"], re.sub(r"\s+", " ", html[m.start() : m.start() + 300]), m
    return None, None, None


def show_example(title, pattern, chunk_filter):
    slug, snippet, m = first_article_matching(pattern)
    print(f"=== {title} ({slug})")
    print("HTML:", snippet)
    for c in chunks_df.filter(F.col("slug") == slug).filter(chunk_filter(m)).orderBy("chunk_id").limit(2).collect():
        print(f"CHUNK {c['chunk_id']} [{c['tag']}] section={c['section']!r}: {c['text'][:200]}")


# Show the raw HTML for each structural loss next to the chunks from the same article
show_example("List", r"<li\b.*?</li>", lambda m: F.col("tag") == "li")
show_example("Link", r"<a\s[^>]*>([^<]+)</a>", lambda m: F.col("text").contains(m.group(1)))
show_example("Heading level", r"<h3>([^<]+)</h3>", lambda m: F.col("section") == m.group(1))

In [ ]:
# Chunks with no block tag, so the source element is not yet identified
print("=== Chunks with tag 'text' (no block tag)")
for c in chunks_df.filter(F.col("tag") == "text").orderBy("chunk_id").limit(3).collect():
    print(f"CHUNK {c['chunk_id']} section={c['section']!r}: {c['text'][:200]}")